# Save Your Work

Before you begin, save a copy of this notebook to your Google Drive: **File > Save a copy in Drive**.

# Module 14 Assessment — Deep Learning (Solution)

Build and compare deep learning architectures on Fashion-MNIST: a dense baseline, a CNN, and a regularized CNN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32")  / 255.0

# Class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {class_names}")
# Train: (60000, 28, 28), Test: (10000, 28, 28)
# Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## Part 1: Dense Baseline

### Task 1: Build and Train a Dense Network

In [ ]:
# Build dense model
model_dense = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_dense.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_dense = model_dense.fit(
    X_train, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=64,
    verbose=1
)

test_loss, test_acc = model_dense.evaluate(X_test, y_test, verbose=0)
print(f"Dense baseline test accuracy: {test_acc:.4f}")
# Dense baseline test accuracy: ~0.8800

### Task 2: Training Curves

In [ ]:
# Plot accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history_dense.history['accuracy'], label='Train')
axes[0].plot(history_dense.history['val_accuracy'], label='Validation')
axes[0].set_title('Dense Baseline — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss
axes[1].plot(history_dense.history['loss'], label='Train')
axes[1].plot(history_dense.history['val_loss'], label='Validation')
axes[1].set_title('Dense Baseline — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

The model begins to overfit around epoch 8–10, where training accuracy continues to rise while validation accuracy plateaus or slightly decreases. The growing gap between training loss and validation loss after epoch 10 confirms that the model is memorizing training patterns rather than learning generalizable features.

## Part 2: CNN Architecture

### Task 3: Build a CNN

In [ ]:
X_train_cnn = X_train[..., np.newaxis]  # shape (60000, 28, 28, 1)
X_test_cnn  = X_test[..., np.newaxis]

# Build CNN model
model_cnn = keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_cnn = model_cnn.fit(
    X_train_cnn, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=64,
    verbose=1
)

test_loss_cnn, test_acc_cnn = model_cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f"CNN test accuracy: {test_acc_cnn:.4f}")
# CNN test accuracy: ~0.9200
print(f"Improvement over dense baseline: {test_acc_cnn - test_acc:+.4f}")

### Task 4: Apply Regularization

In [ ]:
# Regularized CNN
model_reg = keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model_reg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_reg = model_reg.fit(
    X_train_cnn, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

test_loss_reg, test_acc_reg = model_reg.evaluate(X_test_cnn, y_test, verbose=0)
epochs_trained = len(history_reg.history['loss'])
print(f"Regularized CNN test accuracy: {test_acc_reg:.4f}")
print(f"Training stopped at epoch: {epochs_trained}")
# Regularized CNN test accuracy: ~0.9300
# Training stopped at epoch: ~10-12

## Part 3: Analysis

### Task 5: Confusion Matrix

In [ ]:
# Generate predictions
y_pred = np.argmax(model_reg.predict(X_test_cnn), axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display with class labels
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
plt.title('Regularized CNN — Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Find most confused pairs (off-diagonal maxima)
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

# Top 3 confused pairs
print("Most confused class pairs:")
for _ in range(3):
    i, j = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
    print(f"  {class_names[i]} predicted as {class_names[j]}: {cm_no_diag[i, j]} times")
    cm_no_diag[i, j] = 0

# Most confused pairs: Shirt/T-shirt, Pullover/Coat, Shirt/Coat

The most commonly confused pairs are Shirt with T-shirt/top, and Pullover with Coat, which is expected because these garment types share similar overall shapes, textures, and color distributions in grayscale 28x28 images. Without color information and at low resolution, distinguishing fine structural details like collar style or sleeve cut that differentiate these categories is extremely difficult for the model.

### Task 6: Comparison Table

| Model | Test Accuracy | Epochs Trained |
|-------|--------------|----------------|
| Dense baseline | ~88% | 20 |
| CNN (no regularization) | ~92% | 20 |
| CNN + Dropout + BatchNorm + EarlyStopping | ~93% | ~10–12 |

### Task 7: Reflection

**1. Why does the CNN outperform the dense network on image data?**

CNNs exploit the spatial structure of images through convolutional filters that detect local features (edges, textures, shapes) while sharing weights across all spatial positions. Dense networks treat every pixel independently and cannot capture spatial hierarchies, leading them to learn far more parameters with less inductive bias about how images are structured.

**2. What does Dropout prevent during training?**

Dropout randomly deactivates a fraction of neurons during each training step, preventing any single neuron from becoming overly reliant on specific co-activations. This forces the network to learn redundant, distributed representations, which reduces overfitting and improves generalization to unseen data.

**3. When would you prefer EarlyStopping over training for a fixed number of epochs?**

EarlyStopping is preferable when you do not know in advance how many epochs are needed, when training is expensive and you want to avoid wasting compute on epochs that no longer improve validation performance, or when you want an automatic guard against overfitting without manually tuning the epoch count for each experiment.